# Trabalho 11ABDR — Análise de jogos internacionais de futebol com Apache Spark

**Disciplina:** Distributed Data Processing e Storage — MBA Engenharia de Dados (FIAP)

**Integrantes:**
- Jonathan Gamez Tomoyosi — 378455
- Leonardo da Silva — 375821

Dataset: `results_11ABDR.csv` (Kaggle) — resultados de jogos internacionais masculinos de futebol.
Todas as respostas foram obtidas com **PySpark** (DataFrame API) e conferidas com **Spark SQL**. Não foi usado pandas.

## Preparação: sessão Spark e importação do arquivo

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (StructType, StructField, StringType,
                               IntegerType, DateType, BooleanType)

spark = SparkSession.builder.appName("Trabalho_11ABDR").getOrCreate()
spark

O cabeçalho original do arquivo vem com sufixos do Kaggle (`home_teamName`, `home_scoreHome`, ...).
Por isso o arquivo é lido com um **schema explícito**, usando os nomes do enunciado e os tipos corretos
(placares como inteiro, data como data).

In [2]:
schema = StructType([
    StructField("date",       DateType(),    True),
    StructField("home_team",  StringType(),  True),
    StructField("away_team",  StringType(),  True),
    StructField("home_score", IntegerType(), True),
    StructField("away_score", IntegerType(), True),
    StructField("tournament", StringType(),  True),
    StructField("city",       StringType(),  True),
    StructField("country",    StringType(),  True),
    StructField("neutral",    BooleanType(), True),   # desconsiderar (conforme enunciado)
])

df = (spark.read
      .option("header", True)
      .option("encoding", "UTF-8")
      .schema(schema)
      .csv("db/results_11ABDR.csv"))

df.printSchema()
df.show(5, truncate=False)

# View temporária para as consultas em Spark SQL
df.createOrReplaceTempView("results")

root
 |-- date: date (nullable = true)
 |-- home_team: string (nullable = true)
 |-- away_team: string (nullable = true)
 |-- home_score: integer (nullable = true)
 |-- away_score: integer (nullable = true)
 |-- tournament: string (nullable = true)
 |-- city: string (nullable = true)
 |-- country: string (nullable = true)
 |-- neutral: boolean (nullable = true)



+----------+---------+---------+----------+----------+----------+-------+--------+-------+
|date      |home_team|away_team|home_score|away_score|tournament|city   |country |neutral|
+----------+---------+---------+----------+----------+----------+-------+--------+-------+
|1872-11-30|Scotland |England  |0         |0         |Friendly  |Glasgow|Scotland|false  |
|1873-03-08|England  |Scotland |4         |2         |Friendly  |London |England |false  |
|1874-03-07|Scotland |England  |2         |1         |Friendly  |Glasgow|Scotland|false  |
|1875-03-06|England  |Scotland |2         |2         |Friendly  |London |England |false  |
|1876-03-04|Scotland |England  |3         |0         |Friendly  |Glasgow|Scotland|false  |
+----------+---------+---------+----------+----------+----------+-------+--------+-------+
only showing top 5 rows


Conferência de qualidade: valores nulos por coluna (placares nulos afetariam as contagens de vitória/empate).

In [3]:
df.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in df.columns]).show()

+----+---------+---------+----------+----------+----------+----+-------+-------+
|date|home_team|away_team|home_score|away_score|tournament|city|country|neutral|
+----+---------+---------+----------+----------+----------+----+-------+-------+
|   0|        0|        0|         0|         0|         0|   0|      0|      0|
+----+---------+---------+----------+----------+----------+----+-------+-------+



## 1 - Quantos registros existem na base?

In [4]:
total_registros = df.count()
print(f"Total de registros: {total_registros}")

Total de registros: 40839


In [5]:
spark.sql("SELECT COUNT(*) AS total_registros FROM results").show()

+---------------+
|total_registros|
+---------------+
|          40839|
+---------------+



**Resposta 1:** A base possui **40.839 registros**.

## 2 - Quantas equipes únicas mandantes existem na base?

In [6]:
mandantes_unicos = df.select(F.countDistinct("home_team").alias("mandantes_unicos")).first()[0]
print(f"Equipes mandantes únicas: {mandantes_unicos}")

Equipes mandantes únicas: 309


In [7]:
spark.sql("SELECT COUNT(DISTINCT home_team) AS mandantes_unicos FROM results").show()

+----------------+
|mandantes_unicos|
+----------------+
|             309|
+----------------+



**Resposta 2:** Existem **309 equipes mandantes únicas**.

## 3 - Quantas vezes as equipes mandantes saíram vitoriosas?

In [8]:
vitorias_mandante = df.filter(F.col("home_score") > F.col("away_score")).count()
print(f"Vitórias dos mandantes: {vitorias_mandante}")

Vitórias dos mandantes: 19864


In [9]:
spark.sql("SELECT COUNT(*) AS vitorias_mandante FROM results WHERE home_score > away_score").show()

+-----------------+
|vitorias_mandante|
+-----------------+
|            19864|
+-----------------+



**Resposta 3:** As equipes mandantes venceram **19.864 partidas**.

## 4 - Quantas vezes as equipes visitantes saíram vitoriosas?

In [10]:
vitorias_visitante = df.filter(F.col("away_score") > F.col("home_score")).count()
print(f"Vitórias dos visitantes: {vitorias_visitante}")

Vitórias dos visitantes: 11544


In [11]:
spark.sql("SELECT COUNT(*) AS vitorias_visitante FROM results WHERE away_score > home_score").show()

+------------------+
|vitorias_visitante|
+------------------+
|             11544|
+------------------+



**Resposta 4:** As equipes visitantes venceram **11.544 partidas**.

## 5 - Quantas partidas resultaram em empate?

In [12]:
empates = df.filter(F.col("home_score") == F.col("away_score")).count()
print(f"Empates: {empates}")

# Conferência: vitórias mandante + vitórias visitante + empates = total de registros
print(f"Conferência: {vitorias_mandante} + {vitorias_visitante} + {empates} = "
      f"{vitorias_mandante + vitorias_visitante + empates} (total: {total_registros})")

Empates: 9431
Conferência: 19864 + 11544 + 9431 = 40839 (total: 40839)


In [13]:
spark.sql("SELECT COUNT(*) AS empates FROM results WHERE home_score = away_score").show()

+-------+
|empates|
+-------+
|   9431|
+-------+



**Resposta 5:** **9.431 partidas** terminaram empatadas (19.864 + 11.544 + 9.431 = 40.839, fechando com o total da base).

## 6 - Quantas partidas foram realizadas em cada país?

In [14]:
partidas_por_pais = (df.groupBy("country")
                       .agg(F.count("*").alias("partidas"))
                       .orderBy(F.desc("partidas"), F.asc("country")))

print(f"Quantidade de países que sediaram partidas: {partidas_por_pais.count()}")
partidas_por_pais.show(partidas_por_pais.count(), truncate=False)

Quantidade de países que sediaram partidas: 266


+--------------------------------+--------+
|country                         |partidas|
+--------------------------------+--------+
|United States                   |1144    |
|France                          |801     |
|England                         |687     |
|Malaysia                        |644     |
|Sweden                          |637     |
|Germany                         |581     |
|Brazil                          |529     |
|Spain                           |517     |
|Thailand                        |483     |
|Italy                           |480     |
|Switzerland                     |477     |
|Austria                         |475     |
|United Arab Emirates            |472     |
|South Africa                    |470     |
|Qatar                           |467     |
|South Korea                     |453     |
|Argentina                       |449     |
|Hungary                         |431     |
|Chile                           |405     |
|Belgium                        

In [15]:
spark.sql("""
    SELECT country, COUNT(*) AS partidas
    FROM results
    GROUP BY country
    ORDER BY partidas DESC, country
""").show(10, truncate=False)

+-------------+--------+
|country      |partidas|
+-------------+--------+
|United States|1144    |
|France       |801     |
|England      |687     |
|Malaysia     |644     |
|Sweden       |637     |
|Germany      |581     |
|Brazil       |529     |
|Spain        |517     |
|Thailand     |483     |
|Italy        |480     |
+-------------+--------+
only showing top 10 rows


**Resposta 6:** As partidas foram realizadas em **266 países**; a contagem de cada um está na tabela acima (lista completa). Os que mais sediaram: United States (1.144), France (801), England (687), Malaysia (644) e Sweden (637).

## 7 - Qual país teve mais partidas?

In [16]:
max_partidas = partidas_por_pais.agg(F.max("partidas")).first()[0]
# filtra pelo máximo para exibir eventuais empates na primeira posição
partidas_por_pais.filter(F.col("partidas") == max_partidas).show(truncate=False)

+-------------+--------+
|country      |partidas|
+-------------+--------+
|United States|1144    |
+-------------+--------+



In [17]:
spark.sql("""
    SELECT country, COUNT(*) AS partidas
    FROM results
    GROUP BY country
    ORDER BY partidas DESC
    LIMIT 1
""").show(truncate=False)

+-------------+--------+
|country      |partidas|
+-------------+--------+
|United States|1144    |
+-------------+--------+



**Resposta 7:** O país com mais partidas foi os **Estados Unidos (United States), com 1.144 partidas**.

## 8 - Qual a partida com maior número de gols?

In [18]:
df_gols = df.withColumn("total_gols", F.col("home_score") + F.col("away_score"))
max_gols = df_gols.agg(F.max("total_gols")).first()[0]

# filtra pelo máximo para exibir todas as partidas empatadas no topo, se houver
(df_gols.filter(F.col("total_gols") == max_gols)
        .select("date", "home_team", "away_team", "home_score", "away_score",
                "total_gols", "tournament", "city", "country")
        .show(truncate=False))

+----------+---------+--------------+----------+----------+----------+----------------------------+-------------+---------+
|date      |home_team|away_team     |home_score|away_score|total_gols|tournament                  |city         |country  |
+----------+---------+--------------+----------+----------+----------+----------------------------+-------------+---------+
|2001-04-11|Australia|American Samoa|31        |0         |31        |FIFA World Cup qualification|Coffs Harbour|Australia|
+----------+---------+--------------+----------+----------+----------+----------------------------+-------------+---------+



In [19]:
spark.sql("""
    SELECT date, home_team, away_team, home_score, away_score,
           home_score + away_score AS total_gols, tournament, city, country
    FROM results
    WHERE home_score + away_score = (SELECT MAX(home_score + away_score) FROM results)
""").show(truncate=False)

+----------+---------+--------------+----------+----------+----------+----------------------------+-------------+---------+
|date      |home_team|away_team     |home_score|away_score|total_gols|tournament                  |city         |country  |
+----------+---------+--------------+----------+----------+----------+----------------------------+-------------+---------+
|2001-04-11|Australia|American Samoa|31        |0         |31        |FIFA World Cup qualification|Coffs Harbour|Australia|
+----------+---------+--------------+----------+----------+----------+----------------------------+-------------+---------+



**Resposta 8:** **Australia 31 x 0 American Samoa**, em 11/04/2001, pelas Eliminatórias da Copa do Mundo (FIFA World Cup qualification), em Coffs Harbour (Austrália) — **31 gols** no total.

## 9 - Qual a maior goleada?

Goleada medida pela **diferença de gols** entre as equipes (em módulo, vale para mandante ou visitante).

In [20]:
df_saldo = df.withColumn("diferenca_gols", F.abs(F.col("home_score") - F.col("away_score")))
max_diferenca = df_saldo.agg(F.max("diferenca_gols")).first()[0]

(df_saldo.filter(F.col("diferenca_gols") == max_diferenca)
         .select("date", "home_team", "away_team", "home_score", "away_score",
                 "diferenca_gols", "tournament", "city", "country")
         .show(truncate=False))

+----------+---------+--------------+----------+----------+--------------+----------------------------+-------------+---------+
|date      |home_team|away_team     |home_score|away_score|diferenca_gols|tournament                  |city         |country  |
+----------+---------+--------------+----------+----------+--------------+----------------------------+-------------+---------+
|2001-04-11|Australia|American Samoa|31        |0         |31            |FIFA World Cup qualification|Coffs Harbour|Australia|
+----------+---------+--------------+----------+----------+--------------+----------------------------+-------------+---------+



In [21]:
spark.sql("""
    SELECT date, home_team, away_team, home_score, away_score,
           ABS(home_score - away_score) AS diferenca_gols, tournament, city, country
    FROM results
    WHERE ABS(home_score - away_score) = (SELECT MAX(ABS(home_score - away_score)) FROM results)
""").show(truncate=False)

+----------+---------+--------------+----------+----------+--------------+----------------------------+-------------+---------+
|date      |home_team|away_team     |home_score|away_score|diferenca_gols|tournament                  |city         |country  |
+----------+---------+--------------+----------+----------+--------------+----------------------------+-------------+---------+
|2001-04-11|Australia|American Samoa|31        |0         |31            |FIFA World Cup qualification|Coffs Harbour|Australia|
+----------+---------+--------------+----------+----------+--------------+----------------------------+-------------+---------+



**Resposta 9:** A maior goleada também foi **Australia 31 x 0 American Samoa** (11/04/2001, FIFA World Cup qualification, Coffs Harbour, Austrália), com **diferença de 31 gols**.

## 10 - Quantos jogos ocorreram no Brasil?

Considera-se o país onde a partida foi realizada (coluna `country` = `Brazil`).

In [22]:
jogos_brasil = df.filter(F.col("country") == "Brazil").count()
print(f"Jogos realizados no Brasil: {jogos_brasil}")

Jogos realizados no Brasil: 529


In [23]:
spark.sql("SELECT COUNT(*) AS jogos_no_brasil FROM results WHERE country = 'Brazil'").show()

+---------------+
|jogos_no_brasil|
+---------------+
|            529|
+---------------+



**Resposta 10:** Ocorreram **529 jogos no Brasil** (partidas cujo país-sede é o Brasil).

## Encerramento

In [24]:
spark.stop()